#Great Expectations Utility File
> From my previous observations about the data, I made this file to check for the data that I observed.
> Initally all my bronze checks should fail, BUT
> If I my silver transforms correctly I will be able to to pass all the checks.

In [0]:
%pip install great_expectations

In [0]:
import great_expectations as gx
import great_expectations.expectations as gxe
import pandas as pd
from itertools import chain

In [0]:
def validate_table(table_name, df):
    # 1. Convert Spark to Pandas for the GXE processing
    pdf = df.toPandas()
    context = gx.get_context()

    # 2. Initialize the Batch
    batch = context.data_sources.pandas_default.read_dataframe(pdf, asset_name=table_name)

    expectations = []
    failed_checks = []

    # ── circuits ──────────────────────────────────────────────────────────────
    # Silver checks:
    #   - country expanded to long form (USA -> United States, UK -> United Kingdom, UAE -> United Arab Emirates)
    #   - url removed (checked globally below)
    if table_name == "race_data_circuits":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="circuitid"),
            gxe.ExpectColumnValuesToNotBeNull(column="country"),
            gxe.ExpectColumnValuesToBeInSet(column="country", value_set=[
                "Australia", "Malaysia", "Bahrain", "Spain", "Monaco",
                "Turkey", "Canada", "United States", "France", "Germany",
                "Hungary", "Belgium", "Italy", "Singapore", "Japan",
                "China", "Brazil", "United Arab Emirates", "United Kingdom",
                "Argentina", "Mexico", "Portugal", "Morocco", "Netherlands",
                "Austria", "Sweden", "Switzerland", "South Africa", "South Korea",
                "India", "Russia", "Azerbaijan", "Saudi Arabia", "Qatar", "Vietnam"
            ]),
        ]

    # ── constructor_results ───────────────────────────────────────────────────
    # Silver checks:
    #   - status \N -> null handled by to_null(), D -> "Disqualified" by constructor_results_specific()
    #   - so we check status is either null or a known full-form value, not raw "D" or "\N"
    elif table_name == "race_data_constructor_results":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="constructorid"),
            gxe.ExpectColumnValuesToBeInSet(column="status", value_set=[
                "Disqualified", None
            ]),
        ]

    # ── constructor_standings ─────────────────────────────────────────────────
    # Silver checks:
    #   - position \N -> null by to_null(), then cast to int by cast_to_integer()
    #   - positionText "-" -> "None", "E" -> "Excluded" by position_text_fix()
    elif table_name == "race_data_constructor_standings":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="constructorid"),
            gxe.ExpectColumnValuesToBeOfType(column="position", type_="int64"),
            gxe.ExpectColumnValuesToBeInSet(column="positionText", value_set=[
                "Excluded", "None", None
            ]),
        ]

    # ── constructors ──────────────────────────────────────────────────────────
    # Silver checks:
    #   - url removed (checked globally below)
    elif table_name == "race_data_constructors":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="constructorid"),
            gxe.ExpectColumnValuesToNotBeNull(column="name"),
        ]

    # ── driver_standings ──────────────────────────────────────────────────────
    # Silver checks:
    #   - position \N -> null by to_null(), then cast to int by cast_to_integer()
    #   - positionText "-" -> "None", "E" -> "Excluded" by position_text_fix()
    elif table_name == "race_data_driver_standings":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="driverid"),
            gxe.ExpectColumnValuesToBeOfType(column="position", type_="int64"),
            gxe.ExpectColumnValuesToBeInSet(column="positionText", value_set=[
                "Excluded", "None", None
            ]),
        ]

    # ── drivers ───────────────────────────────────────────────────────────────
    # Silver checks:
    #   - number \N -> null by to_null(), then cast to int by cast_to_integer()
    #   - forename/surname trimmed by drivers_specific() and trim_strings()
    #   - dob cast to date by cast_date_columns()
    #   - url removed (checked globally below)
    elif table_name == "race_data_drivers":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="driverid"),
            gxe.ExpectColumnValuesToNotBeNull(column="forename"),
            gxe.ExpectColumnValuesToNotBeNull(column="surname"),
            # After trim, no leading/trailing whitespace
            gxe.ExpectColumnValuesToMatchRegex(column="forename", regex=r"^\S.*\S$|^\S$"),
            gxe.ExpectColumnValuesToMatchRegex(column="surname", regex=r"^\S.*\S$|^\S$"),
            # number is null or a valid int (no \N strings remaining)
            gxe.ExpectColumnValuesToBeOfType(column="number", type_="float64"),
            # dob is a valid yyyy-MM-dd date string after cast_date_columns()
            gxe.ExpectColumnValuesToMatchRegex(column="dob", regex=r"^\d{4}-\d{2}-\d{2}$"),
        ]

    # ── lap_times ─────────────────────────────────────────────────────────────
    # Silver checks:
    #   - time col dropped by transform_time_to_precision() in favour of milliseconds
    #   - milliseconds cast to int
    elif table_name == "race_data_lap_times":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="driverid"),
            gxe.ExpectColumnValuesToNotBeNull(column="lap"),
            gxe.ExpectColumnValuesToBeBetween(column="lap", min_value=1, max_value=100),
            gxe.ExpectColumnValuesToBeOfType(column="milliseconds", type_="float64"),
            gxe.ExpectColumnValuesToNotBeNull(column="milliseconds"),
        ]

    # ── pit_stops ─────────────────────────────────────────────────────────────
    # Silver checks:
    #   - duration dropped, time dropped by transform_time_to_precision() in favour of milliseconds
    #   - milliseconds cast to int
    elif table_name == "race_data_pit_stops":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="driverid"),
            gxe.ExpectColumnValuesToNotBeNull(column="stop"),
            gxe.ExpectColumnValuesToNotBeNull(column="lap"),
            gxe.ExpectColumnValuesToBeOfType(column="milliseconds", type_="float64"),
        ]

    # ── qualifying ────────────────────────────────────────────────────────────
    # Silver checks:
    #   - q1/q2/q3 converted to integer milliseconds by transform_time_to_precision()
    elif table_name == "race_data_qualifying":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="driverid"),
            gxe.ExpectColumnValuesToNotBeNull(column="constructorid"),
            gxe.ExpectColumnValuesToBeOfType(column="q1", type_="float64"),
            gxe.ExpectColumnValuesToBeOfType(column="q2", type_="float64"),
            gxe.ExpectColumnValuesToBeOfType(column="q3", type_="float64"),
        ]

    # ── races ─────────────────────────────────────────────────────────────────
    # Silver checks:
    #   - year dropped by races_specific()
    #   - date cast from string yyyy-MM-dd to date by cast_date_columns()
    #   - all *_date columns cast to date, all *_time columns converted to ms by transform_time_to_precision()
    #   - url removed (checked globally below)
    elif table_name == "race_data_races":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="date"),
            gxe.ExpectColumnValuesToNotBeNull(column="circuitid"),
            gxe.ExpectColumnValuesToMatchRegex(column="date", regex=r"^\d{4}-\d{2}-\d{2}$"),
            # year column should no longer exist after races_specific() drops it
            # (validated via global column existence check below)
            # All *_date columns should match yyyy-MM-dd
            gxe.ExpectColumnValuesToMatchRegex(column="fp1_date",    regex=r"^\d{4}-\d{2}-\d{2}$"),
            gxe.ExpectColumnValuesToMatchRegex(column="fp2_date",    regex=r"^\d{4}-\d{2}-\d{2}$"),
            gxe.ExpectColumnValuesToMatchRegex(column="fp3_date",    regex=r"^\d{4}-\d{2}-\d{2}$"),
            gxe.ExpectColumnValuesToMatchRegex(column="quali_date",  regex=r"^\d{4}-\d{2}-\d{2}$"),
            gxe.ExpectColumnValuesToMatchRegex(column="sprint_date", regex=r"^\d{4}-\d{2}-\d{2}$"),
        ]

    # ── results ───────────────────────────────────────────────────────────────
    # Silver checks:
    #   - number/grid/position/fastestlap/rank: \N -> null then cast to int
    #   - positionText: R/D/E/F/N/W/- expanded to full form by position_text_fix()
    #   - milliseconds: \N -> null then cast to int
    #   - fastestlaptime: converted to ms int by transform_time_to_precision()
    #   - fastestlapspeed: cast to DoubleType by results_specific()
    #   - time col: dropped in favour of milliseconds by transform_time_to_precision()
    elif table_name == "race_data_results":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="resultid"),
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="driverid"),
            gxe.ExpectColumnValuesToNotBeNull(column="constructorid"),
            gxe.ExpectColumnValuesToBeBetween(column="positionorder", min_value=1, max_value=20),
            # All these should be int or null — no \N strings remaining
            gxe.ExpectColumnValuesToBeOfType(column="number",     type_="float64"),
            gxe.ExpectColumnValuesToBeOfType(column="grid",       type_="float64"),
            gxe.ExpectColumnValuesToBeOfType(column="position",   type_="float64"),
            gxe.ExpectColumnValuesToBeOfType(column="rank",       type_="float64"),
            gxe.ExpectColumnValuesToBeOfType(column="milliseconds", type_="float64"),
            # positionText full-form values only — including "F" (Failed to qualify)
            gxe.ExpectColumnValuesToBeInSet(column="positionText", value_set=[
                "Retired", "Disqualified", "Excluded", "Not Classified",
                "Withdrawn", "None", "Failed to qualify", None
            ]),
            # fastestlapspeed should be a double now
            gxe.ExpectColumnValuesToBeOfType(column="fastestLapSpeed", type_="float64"),
        ]

    # ── seasons ───────────────────────────────────────────────────────────────
    # Silver checks:
    #   - year cast to int by season_specific()
    #   - url removed (checked globally below)
    elif table_name == "race_data_seasons":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="year"),
            gxe.ExpectColumnValuesToBeOfType(column="year", type_="int64"),
            gxe.ExpectColumnValuesToBeBetween(column="year", min_value=1950, max_value=2026),
        ]

    # ── sprint_results ────────────────────────────────────────────────────────
    # Silver checks:
    #   - position: \N -> null then cast to int
    #   - positionText: N/R/W expanded to full form by position_text_fix()
    #   - milliseconds: \N -> null then cast to int
    #   - fastestlaptime: converted to ms int by transform_time_to_precision()
    #   - rank: \N -> null then cast to int
    #   - time col dropped in favour of milliseconds by transform_time_to_precision()
    elif table_name == "race_data_sprint_results":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="raceid"),
            gxe.ExpectColumnValuesToNotBeNull(column="driverid"),
            gxe.ExpectColumnValuesToNotBeNull(column="constructorid"),
            gxe.ExpectColumnValuesToBeOfType(column="position",     type_="float64"),
            gxe.ExpectColumnValuesToBeOfType(column="milliseconds", type_="float64"),
            gxe.ExpectColumnValuesToBeOfType(column="rank",         type_="float64"),
            # positionText full-form values only
            gxe.ExpectColumnValuesToBeInSet(column="positionText", value_set=[
                "Not Classified", "Retired", "Withdrawn", None
            ]),
        ]

    # ── status ────────────────────────────────────────────────────────────────
    # Silver checks:
    #   - status kept as-is (some are "+1 Lap", some are "Collision" etc.)
    elif table_name == "race_data_status":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="statusid"),
            gxe.ExpectColumnValuesToNotBeNull(column="status"),
        ]

    # ── fatal accidents drivers ───────────────────────────────────────────────
    # Silver checks:
    #   - date_of_accident converted from M/d/yy -> yyyy-MM-dd by cast_date_columns()
    elif table_name == "race_events_fatal_accidents_drivers":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="driver"),
            gxe.ExpectColumnValuesToMatchRegex(column="date_of_accident", regex=r"^\d{4}-\d{2}-\d{2}$"),
        ]

    # ── fatal accidents marshalls ─────────────────────────────────────────────
    # Silver checks:
    #   - date_of_accident converted from M/d/yy -> yyyy-MM-dd by cast_date_columns()
    elif table_name == "race_events_fatal_accidents_marshalls":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="name"),
            gxe.ExpectColumnValuesToMatchRegex(column="date_of_accident", regex=r"^\d{4}-\d{2}-\d{2}$"),
        ]

    # ── red flags ─────────────────────────────────────────────────────────────
    # Silver checks:
    #   - resumed expanded by red_flags_specific():
    #       R -> "Resumed", S -> "Standing Start", Y -> "Rolling Start", N -> "Not Resumed"
    #   NOTE: value_set here now matches the transform output, not the old bronze guess
    elif table_name == "race_events_red_flags":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="race"),
            gxe.ExpectColumnValuesToBeInSet(column="resumed", value_set=[
                "Resumed", "Standing Start", "Rolling Start", "Not Resumed", None
            ]),
        ]

    # ── safety cars ───────────────────────────────────────────────────────────
    # Silver checks:
    #   - cause normalised by safety_car_specific():
    #       accident*/rain variants -> "accident"
    #       stranded car* variants -> "stranded car"
    #       "multiple accidents" kept as-is
    #       everything else kept as-is
    elif table_name == "race_events_safety_cars":
        expectations += [
            gxe.ExpectColumnValuesToNotBeNull(column="race"),
            gxe.ExpectColumnValuesToNotBeNull(column="cause"),
            gxe.ExpectColumnValuesToNotBeNull(column="deployed"),
            # After normalisation, no raw "accident/rain" or "accidents" variants should remain
            gxe.ExpectColumnValuesToNotMatchRegex(column="cause", regex=r"(?i)accident[s/]|stranded car/"),
        ]

    # ── GLOBAL CHECKS ─────────────────────────────────────────────────────────

    # 1. url column should have been dropped by drop_url() for all tables that had it
    #    (circuits, constructors, drivers, races, seasons)
    if "url" in pdf.columns:
        failed_checks.append("ExpectColumnToNotExist — url column still present, drop_url() may not have run")

    # 2. year column should have been dropped by races_specific() for races table only
    if table_name == "race_data_races" and "year" in pdf.columns:
        failed_checks.append("ExpectColumnToNotExist — year column still present in races, races_specific() may not have run")

    # ── RUN EXPECTATIONS ──────────────────────────────────────────────────────
    for exp in expectations:
        try:
            result = batch.validate(expect=exp)
            if not result.success:
                col_name = getattr(exp, "column", "Table-Level")

                stats = result.result
                unexpected = stats.get("partial_unexpected_list", [])

                msg = f"{exp.__class__.__name__} on '{col_name}'"
                if unexpected:
                    examples = ", ".join([f"'{str(x)}'" for x in unexpected[:3]])
                    msg += f" | Found: {examples}"

                failed_checks.append(msg)
        except Exception as e:
            failed_checks.append(f"ExecutionError on {exp.__class__.__name__}: {str(e)}")

    # CRITICAL: This return must be at the very bottom,
    # aligned with the start of the function code.
    passed = len(failed_checks) == 0
    return passed, failed_checks